# GLiNER Fine-Tuning on Google Colab

## Assumptions and scope
- Input dataset: `combined_output.jsonl`.
- The uploaded dataset is NER-only: each record contains `tokenized_text` and `ner`.
- Entity spans use inclusive token indices, which matches GLiNER's training format.
- This notebook fine-tunes **GLiNER**. The installed `glirel>=0.1` package is checked, but GLiREL is **not fine-tuned from this dataset** because the dataset contains no `relations` field. A GLiREL training set must contain relation annotations.
- The split is deterministic: 80% train, 10% validation, 10% test.

GLiNER's documented training format uses `tokenized_text` plus `[start, end, label]` NER spans. GLiREL's training format additionally requires relation annotations. 

In [ ]:
!pip -q install -U "gliner>=0.2" "glirel>=0.1" scikit-learn

## 1. Upload the dataset

Run this cell and upload `combined_output.jsonl` from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print(DATA_PATH)

In [ ]:
import json
from collections import Counter

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

assert data, "Dataset is empty."
assert all("tokenized_text" in x and "ner" in x for x in data), \
    "Expected tokenized_text and ner in every record."

labels = Counter()
for item in data:
    n_tokens = len(item["tokenized_text"])
    for span in item["ner"]:
        assert len(span) == 3, f"Bad NER span: {span}"
        start, end, label = span
        assert 0 <= start <= end < n_tokens, f"Invalid span: {span}"
        labels[label] += 1

print(f"Examples: {len(data):,}")
print(f"Entities: {sum(labels.values()):,}")
print("Labels:")
for label, count in labels.most_common():
    print(f"  {label:12s} {count:,}")

## 2. Deterministic train/validation/test split

No examples are duplicated across the three splits.

In [ ]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(
    data,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print(f"train: {len(train_data):,}")
print(f"validation: {len(val_data):,}")
print(f"test: {len(test_data):,}")

## 3. Load a GLiNER checkpoint

Start with the small model so the notebook is practical on a free Colab GPU. If it fits your runtime comfortably, you can later change the checkpoint to a larger GLiNER model.

In [ ]:
import torch
from gliner import GLiNER

MODEL_NAME = "urchade/gliner_small-v2.1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

model = GLiNER.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

## 4. Fine-tune GLiNER

The dataset is already in GLiNER's tokenized NER format, so no conversion is needed.

In [ ]:
OUTPUT_DIR = "./gliner_finetuned"

trainer = model.train_model(
    train_dataset=train_data,
    eval_dataset=val_data,
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
)

trainer.save_model()
print("Saved to:", OUTPUT_DIR)

## 5. Load the fine-tuned model


In [ ]:
finetuned_model = GLiNER.from_pretrained(OUTPUT_DIR)
finetuned_model = finetuned_model.to(DEVICE)
print("Fine-tuned model loaded.")

## 6. Inference smoke test


In [ ]:
example = test_data[0]
text = " ".join(example["tokenized_text"])
entity_types = sorted(labels.keys())

predictions = finetuned_model.predict_entities(
    text,
    entity_types,
    threshold=0.5,
)

print(text)
print("\nPredictions:")
for entity in predictions:
    print(entity)

## 7. Basic test-set evaluation

This computes exact entity-span precision, recall, and F1. A prediction is correct only when its entity text span and label exactly match a gold entity.

In [ ]:
from collections import defaultdict

def gold_entities(item):
    tokens = item["tokenized_text"]
    return {
        (start, end, label)
        for start, end, label in item["ner"]
    }

def prediction_to_token_span(item, prediction):
    tokens = item["tokenized_text"]
    target = prediction["text"].split()
    label = prediction["label"]

    for start in range(len(tokens) - len(target) + 1):
        if tokens[start:start + len(target)] == target:
            return (start, start + len(target) - 1, label)
    return None

tp = fp = fn = 0
per_label = defaultdict(lambda: [0, 0, 0])

for item in test_data:
    text = " ".join(item["tokenized_text"])
    predictions = finetuned_model.predict_entities(
        text, entity_types, threshold=0.5
    )
    pred = {
        x for x in (
            prediction_to_token_span(item, p) for p in predictions
        ) if x is not None
    }
    gold = gold_entities(item)

    for span in pred & gold:
        tp += 1
        per_label[span[2]][0] += 1
    for span in pred - gold:
        fp += 1
        per_label[span[2]][1] += 1
    for span in gold - pred:
        fn += 1
        per_label[span[2]][2] += 1

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"Exact span precision: {precision:.4f}")
print(f"Exact span recall:    {recall:.4f}")
print(f"Exact span F1:        {f1:.4f}")

## 8. Per-label F1


In [ ]:
for label in sorted(per_label):
    label_tp, label_fp, label_fn = per_label[label]
    p = label_tp / (label_tp + label_fp) if label_tp + label_fp else 0.0
    r = label_tp / (label_tp + label_fn) if label_tp + label_fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    print(f"{label:12s} P={p:.4f} R={r:.4f} F1={f:.4f}")

## 9. Verify GLiREL dataset compatibility

The current dataset is NER-only. GLiREL fine-tuning requires relation annotations in addition to the entity spans. This cell deliberately stops instead of inventing relations.

In [ ]:
has_relations = any("relations" in item for item in data)

if not has_relations:
    raise RuntimeError(
        "GLiREL fine-tuning cannot be performed from combined_output.jsonl: "
        "the dataset contains NER annotations but no relations annotations. "
        "Provide a relation-annotated dataset to add GLiREL fine-tuning."
    )

## 10. Download the fine-tuned GLiNER model


In [ ]:
!zip -qr gliner_finetuned.zip gliner_finetuned

from google.colab import files
files.download("gliner_finetuned.zip")